# TFMA Analysis — COMP315 Group 5

This notebook loads the latest Evaluator artifact from the integrated TFX pipeline and renders overall and per-slice BinaryAccuracy and AUC.

In [ ]:
from pathlib import Path

import pandas as pd
import tensorflow_model_analysis as tfma

PROJECT_DIR = Path.cwd().resolve()
if PROJECT_DIR.name == 'notebooks':
    PROJECT_DIR = PROJECT_DIR.parent

EVALUATOR_ROOT = (
    PROJECT_DIR / 'pipeline_output' / 'full_pipeline' /
    'Evaluator' / 'evaluation'
)
evaluation_paths = [path for path in EVALUATOR_ROOT.iterdir() if path.is_dir()]
if not evaluation_paths:
    raise FileNotFoundError(f'No Evaluator artifacts found under {EVALUATOR_ROOT}')

EVALUATION_PATH = max(evaluation_paths, key=lambda path: int(path.name))
print('Loading TFMA artifact:', EVALUATION_PATH)

In [ ]:
eval_result = tfma.load_eval_result(str(EVALUATION_PATH))
print('Number of overall and feature-value slices:', len(eval_result.slicing_metrics))

## Interactive slicing visualization

Use the controls below to compare BinaryAccuracy and AUC across the overall dataset and the `marital`, `job`, and `education` slices.

In [ ]:
tfma.view.render_slicing_metrics(eval_result)

## Metric table

The table makes the same TFMA results easier to quote in the written analysis.

In [ ]:
rows = []
for slice_key, metrics in eval_result.slicing_metrics:
    metric_values = metrics['']['']
    if slice_key:
        feature, value = slice_key[0]
        slice_name = f'{feature}={value}'
    else:
        slice_name = 'overall'
    rows.append({
        'slice': slice_name,
        'BinaryAccuracy': metric_values['binary_accuracy']['doubleValue'],
        'AUC': metric_values['auc']['doubleValue'],
    })

slice_metrics = pd.DataFrame(rows).sort_values('slice').reset_index(drop=True)
slice_metrics.style.format({'BinaryAccuracy': '{:.4f}', 'AUC': '{:.4f}'})

## Interpretation of the Airflow run

The latest Airflow evaluation achieved **0.8888 BinaryAccuracy** and **0.8221 AUC** overall. The difference between these metrics matters because accuracy uses a fixed decision threshold, whereas AUC measures how well the model ranks positive examples across thresholds.

For `marital`, accuracy ranged from **0.8690** for single clients to **0.8977** for married clients. AUC ranged from **0.8057** for single clients to **0.8490** for divorced clients. This indicates that performance is not uniform even within this three-value feature.

For `education`, primary education had the highest accuracy (**0.9139**) and AUC (**0.8454**). Tertiary education had the lowest accuracy (**0.8570**), while unknown education had the lowest AUC (**0.7807**).

The largest differences appeared in `job`. Blue-collar clients had the highest accuracy (**0.9326**), while services had the highest AUC (**0.8875**). Student and retired clients had the lowest accuracies, **0.7270** and **0.7526**, respectively; their AUC values were also relatively low at **0.7959** and **0.8002**. These gaps deserve further investigation with slice sizes, positive-label rates, confusion-matrix metrics, and the fairness indicator. Accuracy alone may be affected by class imbalance and different base rates, so these results do not by themselves establish unfair treatment or explain its cause.